# 03 · Dual-output lab (the important one)

**The bug we fixed.** The old pipeline emitted a single MRC PDF whose colour
layers were **JPEG2000 (JPXDecode) at ~26 MP**. JPEG2000-in-PDF decodes slowly
in MuPDF and renders **blank in macOS Preview and Firefox PDF.js** → "white
pages that never load". Measured on real pages (150 dpi render): JPEG2000
**~1600 ms/page** vs JPEG **~210 ms/page** — 7.6× slower — and the file wasn't
even smaller.

**The fix — two outputs from one OCR pass:**

| profile | format | for | speed |
|---|---|---|---|
| `distribution` | **JPEG** (DCTDecode) MRC, bg↓3 fg↓2, linearized | humans, the catalog | fast everywhere |
| `archival` | **JPEG2000** MRC → **PDF/A-2b** | preservation master | slow decode, opened rarely |

Text stays razor-sharp in both: the JBIG2 1-bit mask keeps full resolution;
only the colour background/foreground downsample. This notebook reproduces the
experiment so the settings stay evidence-based.

In [ ]:
import sys
from pathlib import Path

# project importable when the kernel isn't the project .venv
sys.path.insert(0, str(Path.cwd().parent))

from evilflowers_books_digitalizer.runtime import load_runtime
from evilflowers_books_digitalizer.cache import LocalCache

# ONE source of truth for cache_dir + output_dir (paths resolve to the project
# root regardless of the notebook's working directory) — this is the unified
# cache every notebook and the production runner share.
rt = load_runtime()
cache = LocalCache(rt.cache_dir)
print("cache :", rt.cache_dir)
print("output:", rt.output_dir)

In [ ]:
import pymupdf

def show_pdf_page(pdf_path, page=1, dpi=110):
    """Render one PDF page inline (for visual QA)."""
    from IPython.display import Image, display
    d = pymupdf.open(str(pdf_path))
    page = min(page, len(d) - 1)
    display(Image(data=d[page].get_pixmap(dpi=dpi).tobytes("png")))

def page_render_ms(pdf_path, dpi=150):
    """Mean per-page render time (ms) — the viewer-load-speed proxy."""
    import time
    d = pymupdf.open(str(pdf_path))
    ts = []
    for i in range(len(d)):
        t = time.time(); d[i].get_pixmap(dpi=dpi); ts.append((time.time() - t) * 1000)
    return sum(ts) / len(ts)

def cached_books():
    """(source, book_id) for every book staged in the local cache."""
    scans = rt.cache_dir / "scans"
    return sorted((p.parent.name, p.name) for p in scans.glob("*/*") if p.is_dir())

### 1 · Clean pages + one OCR pass
Reuse cached clean pages if present, else clean a small sample.

In [ ]:
import shutil, cv2
from evilflowers_books_digitalizer.pipeline.base import BookContext
from evilflowers_books_digitalizer.pipeline.steps.scantailor import ScanTailorScans
from evilflowers_books_digitalizer.pipeline.steps.ocr import OcrPages

src, bid = cached_books()[0]
frames = cache.list_tiffs(src, bid)
idx = [i for i in (1, 8, len(frames)//3, len(frames)//2, 2*len(frames)//3, len(frames)-3)
       if i < len(frames)]
sample = [frames[i] for i in idx if cv2.imread(str(frames[i])) is not None]

work = rt.cache_dir / "lab" / "dual_work"
out = rt.cache_dir / "lab" / "dual_out"
for d in (work, out):
    if d.exists(): shutil.rmtree(d)
    d.mkdir(parents=True)
ctx = BookContext(source=src, book_id="DUAL", work_dir=work, output_dir=out, tiffs=sample)
ctx = ScanTailorScans(color_mode="mixed", dpi=300, output_dpi=600).run(ctx)
ctx = OcrPages(language="slk+eng", dpi=600).run(ctx)
print("clean+OCR'd", len(ctx.tiffs), "pages; hOCR:", ctx.artifacts["hocr"].name)

### 2 · Sweep encoder profiles
Measure size + per-page decode time for each candidate.

In [ ]:
import time, copy
from evilflowers_books_digitalizer.pipeline.profiles import PdfProfile
from evilflowers_books_digitalizer.pipeline.steps.render import RenderPdf

candidates = [
    PdfProfile("jp2_nodownsample", image_format="jpeg2000"),          # the old broken output
    PdfProfile("jpeg_bg3_fg2", image_format="jpeg", bg_downsample=3, fg_downsample=2),
    PdfProfile("jpeg_bg4", image_format="jpeg", bg_downsample=4),
    PdfProfile("jp2_bg2_archival", image_format="jpeg2000", bg_downsample=2),
]
rows = []
for prof in candidates:
    c = copy.copy(ctx); c.artifacts = dict(ctx.artifacts); c.metadata = dict(ctx.metadata)
    c.metadata.pop("outputs", None)
    RenderPdf([prof], dpi=600).run(c)
    pdf = c.artifacts[f"pdf_{prof.name}"]
    rows.append((prof.name, prof.image_format, c.metadata["outputs"][prof.name]["mb"],
                 round(page_render_ms(pdf))))
print(f"{'profile':22} {'fmt':9} {'MB':>6} {'ms/pg':>7}")
for name, fmt, mb, ms in rows:
    print(f"{name:22} {fmt:9} {mb:6.2f} {ms:7.0f}")

The JPEG profiles decode ~7× faster than JPEG2000 for comparable (or smaller)
size — exactly the production split: `distribution` = JPEG, `archival` = JP2.

### 3 · Produce both production outputs + validate PDF/A

In [ ]:
from evilflowers_books_digitalizer.pipeline.profiles import DISTRIBUTION, ARCHIVAL
from evilflowers_books_digitalizer.pipeline.steps.enrich import EnrichPdfMetadata
from evilflowers_books_digitalizer.pipeline.steps.finalize import FinalizePdf
from evilflowers_books_digitalizer.pipeline.steps.pdfa import EnsurePdfA

ctx.metadata.update(title="Demo Book", authors=["Author A"], language="sk")
ctx.metadata.pop("outputs", None)
for step in [RenderPdf([DISTRIBUTION, ARCHIVAL], dpi=600), EnrichPdfMetadata(),
             FinalizePdf(), EnsurePdfA(validate=True)]:
    ctx = step.run(ctx)
for name, info in ctx.metadata["outputs"].items():
    print(f"{name:13} {info['mb']:5.2f} MB  fmt={info['image_format']:9} pdfa={info['pdfa']}")
print("PDF/A-2b valid:", ctx.metadata.get("pdfa_valid"))
print("bookmarks:", ctx.metadata.get("n_bookmarks"))

### 4 · Visual QA — distribution copy

In [ ]:
show_pdf_page(ctx.artifacts['pdf_distribution'], page=2)

### 5 · Alternative approaches (researched 2026-06; follow-up experiments)

Evidence-based backlog — none adopted yet, ranked by upside:

1. **Surya OCR 2** — Slovak 90.4% / Czech 85.8% vs Tesseract `tessdata_best`
   baseline. ⚠️ weights are OpenRAIL-M (non-commercial above a revenue
   threshold) — resolve licensing before adoption. Until then use
   `tessdata_best` for `slk` (avoid the standard tessdata model).
2. **veraPDF in CI** — already wired via `[render] validate_pdfa`; turn on for
   batch runs as a hard quality gate.
3. **`recode_pdf` `--bg/--fg-compression-flags`** — tune jpegoptim `-S` size
   targets toward ~40–80 KB/page for the distribution copy.
4. **UVDoc dewarp** — SOTA geometry, released code; a cheap dewarp-only stage
   to A/B against DocRes on curled/phone-captured pages.
5. **didjvu / DjVu** — alternative MRC path; only if file size stays a pain.

Each is a drop-in `PipelineStep` or a `PdfProfile`/recode flag away — the
architecture is built so adding an output or stage needs no core changes.